# Library Inclusion and Utilities

<!-- structured-notebook -->
## Notebook Summary
Purpose: clean the processed ProQuest news corpus by checking coverage, removing load artifacts and duplicates, correcting metadata fields, and saving the finalized dataframe used by later analysis notebooks.

Main steps:
- Load the current processed ProQuest export and inspect data quality issues.
- Normalize article text and metadata, then document the manual cleanup decisions.
- Write the cleaned corpus back to `../ProQuest/Processed/full_df.csv`.


In [ ]:
# structured-notebook-bootstrap
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


_repo_root = find_repo_root(Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.append(str(_repo_root))

from src.project_paths import (
    ARXIV_RAW_DIR,
    CHROMA_DIR,
    EXTERNAL_NEWS_DIR,
    GUARDIAN_DATA_DIR,
    LLM_CLASSIFICATION_DIR,
    NEWS_HTML_DIR,
    NEWS_OUTPUT_DIR,
    PREPRINT_RAW_DIR,
    PROQUEST_PROCESSED_DIR,
    PROQUEST_UNPROCESSED_DIR,
    PUBMED_PROCESSED_DIR,
    PUBMED_RAW_DIR,
    PUBLICATIONS_TABLE_DIR,
    REDDIT_DATA_DIR,
    ROOT,
    RQ1_FIGURES_DIR,
    RQ4_PLOTS_DIR,
    TOPIC_MATCHING_DIR,
    YOUTUBE_DATA_DIR,
)


In [1]:
import pandas as pd, re
import itables.options as opt
from itables import show
from src.news.nlp import *
from src.news.data_processor import *

# Load Data

In [2]:
import os

df = pd.DataFrame()
folder_path = PROQUEST_UNPROCESSED_DIR / 'Full'

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)

    # Check if the current entry is a file (not a subdirectory)
    if os.path.isfile(file_path):
        try:
            temp_df = read_file(file_path)
            df = pd.concat([df, temp_df], ignore_index=True)
        except Exception as e:
            print(f"Error reading file {filename}: {e}")

In [4]:
df

,Title,Abstract,Full text,Author,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,Country of publication,Publication subject,ISSN,Language of publication,Document type,ProQuest document ID,Document URL,Last updated
0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,CHRISTINE BLEAKLEY [Eire Region ],None available.,THE ANTI-AGEING DIET Youth-boosting foods t...,None,None,Daily Mail; London (UK),2011,"Dec 31, 2011",You,"Solo Syndication, a division of Associated Ne...",London (UK),United Kingdom,General Interest Periodicals--Great Britain,03077578,English,News,913147557,https://www.proquest.com/newspapers/christine...,2012-10-24
2,Verde Maison finds new home in Quarter,"""[...] with the economy and CityNorth's own f...","After three years at CityNorth, a natural s...","Scott, Eugene",None,"Arizona Republic; Phoenix, Ariz.",2011,"Dec 30, 2011",Scottsdale Republic 8,Gannett Media Corp,"Phoenix, Ariz.",United States,General Interest Periodicals--United States,08928711,English,News,916536245,https://www.proquest.com/newspapers/verde-mai...,2017-11-18
3,Adult zits meet a mighty match: FormulaB sy...,"In case you didn't know, Toronto also has a f...",Every society needs a society dermatologist.\...,"Delap, Leanne",None,"Toronto Star; Toronto, Ont.",2011,"Dec 30, 2011",Living,"Torstar Syndication Services, a Division of T...","Toronto, Ont.",Canada,General Interest Periodicals--Canada,03190781,English,None,913071505,https://www.proquest.com/newspapers/adult-zit...,2017-11-18
4,Kaya goes for makeover in expansion bid,According to retail consultancy firm Technopa...,"Mumbai, Dec. 30 -- Kaya Ltd, a unit of person...",None,Net losses; Corporate planning; Consumer goods,Mint; New Delhi,2011,"Dec 30, 2011",None,HT Digital Streams Limited,New Delhi,India,Business And Economics,None,English,NEWSPAPER,913005622,https://www.proquest.com/newspapers/kaya-goes...,2024-12-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72186,Josiah Zayner: The man who hacked his own DNA,None available.,"New Delhi, Jan. 5 -- In the first week of Oct...",None,Gene therapy; Deoxyribonucleic acid--DNA; Emb...,Mint; New Delhi,2018,"Jan 5, 2018",None,HT Digital Streams Limited,New Delhi,India,Business And Economics,None,English,News,1984703964,https://www.proquest.com/newspapers/josiah-za...,2024-12-11
72187,"Insight No 1/2018, December 28 to January 4",Because the taxpayer poured [euro]80 million ...,IN THIS EDITION:\nNORDICA DOES NOT WANT TO BE...,None,Aviation; Prices; Airlines; Immigration; Nonc...,"Baltic News Service. Insight; Tallinn, Estonia",2018,"Jan 4, 2018",None,AS Eesti Meedia,"Tallinn, Estonia",Estonia,General Interest Periodicals--Estonia,None,English,News,1984052195,https://www.proquest.com/newspapers/insight-n...,2021-07-08
72188,Car advice with Honest John - your questions ...,None available.,"If your car has developed a fault, turn to Ho...",None,Bans; Automobile industry; Gasoline,Telegraph.co.uk; London,2018,"Jan 3, 2018",Cars,Telegraph Media Group Holdings Limited,London,United Kingdom,General Interest Periodicals--Great Britain,None,English,News,1984337719,https://www.proquest.com/newspapers/car-advic...,2019-09-05
72189,A Rush to Get Off the Water Grid: [Dining I...,None available.,"SAN FRANCISCO -- At Rainbow Grocery, a cooper...","Bowles, Nellie",Social networks; Drinking water; Fluorides; F...,"New York Times, Late Edition (East Coast); Ne...",2018,"Jan 3, 2018",D,New York Times Company,"New York, N.Y.",United States,General Interest Periodicals--United States,03624331,English,News,1983556316,https://www.proquest.com/newspapers/rush-get-...,2018-01-03


In [2]:
df = pd.read_csv(PROQUEST_PROCESSED_DIR / 'full_df.csv')

## Check if enough data was downloaded for every year

## Cleaning Load Problems

## Cleaning Full text

In [4]:
df['Full text'] = df['Full text'].str.replace("\n", " ")

In [5]:
col = 'Full text'
mask = (df[col].isna()) | (df[col].str.strip() == '') | (df[col].str.startswith('Not available'))
df[mask]
df = df[~mask]

In [59]:
df['sent_count'] = df['Full text'].apply(lambda x: len(sent_tokenize(x)))

<!-- structured-notebook -->
## Duplicate-Handling Note
The next cells remove near-duplicate articles by deriving a first-sentence key and comparing repeated records. This is the main guard against syndicated copies inflating topic counts later on.


## Duplicate Cleaning

In [55]:
import nltk
# nltk.download('punkt')
# nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize

In [7]:
df["Full text"] = df["Full text"].astype(str)

In [8]:
s = df["Full text"].fillna("").astype(str)
df["first_sentence"] = s.apply(lambda x: sent_tokenize(x)[0] if x.strip() else "")

In [9]:
df = dedup_by_prefix_any(df, by = ["Title", "first_sentence"])

## Year Cleaning

In [10]:
import re
import numpy as np
import pandas as pd

# --- 1) Clean the existing Publication year ---
def normalize_year(val):
    if pd.isna(val):
        return pd.NA
    s = str(val)
    digits_only = re.sub(r'\D', '', s)              # keep only digits
    m = re.search(r'(19|20)\d{2}', digits_only)     # find 4-digit year
    if not m:
        return pd.NA
    y = int(m.group())
    return y if 1900 <= y <= 2100 else pd.NA

df['Publication year'] = df['Publication year'].apply(normalize_year).astype('Int64')

# --- 2) Parse Publication date to datetime ---
# This should handle "Feb 10, 2023", "May 17, 2022", etc.
pub_dt = pd.to_datetime(df['Publication date'], errors='coerce', infer_datetime_format=True)

# --- 3) Fill missing years from parsed dates ---
year_from_date = pub_dt.dt.year.astype('Int64')
df['Publication year'] = df['Publication year'].fillna(year_from_date)

# --- 4) Fallback: regex on Publication date for any still-missing ---
mask_na = df['Publication year'].isna()
if mask_na.any():
    # Extract a 4-digit year (19xx/20xx) anywhere in the string
    year_from_regex = (
        df.loc[mask_na, 'Publication date']
          .astype(str)
          .str.extract(r'(19|20)\d{2}', expand=False)
          .astype('Int64')
    )
    df.loc[mask_na, 'Publication year'] = year_from_regex

# Make sure dtype is Int64 (nullable int)
df['Publication year'] = df['Publication year'].astype('Int64')

# Optional: see what’s left
print("Unique years:", df['Publication year'].unique())
print("Still missing:", df['Publication year'].isna().sum())


Unique years: <IntegerArray>
[2011, 2012, 2013, 2010, <NA>, 2020, 2018, 2024, 2022, 2025, 2016, 2015, 2014,
 2023, 2021, 2017, 2019]
Length: 17, dtype: Int64
Still missing: 7


## Title

In [11]:
mask = df['Title'].isna()
df = df[~mask]

<!-- structured-notebook -->
## Metadata Cleanup Note
The remaining cleanup steps are mostly targeted fixes for fields that are useful downstream, especially publication year, title, subject labels, and broad topic noise flags.


## Subject Cleaning

### Professional baseball

In [12]:
df[df['Subject'].str.contains('Professional baseball', na=False)]

,Title,Abstract,Full text,Author,Subject,Publication title,Publication year,Publication date,Section,Publisher,Place of publication,Country of publication,Publication subject,ISSN,Language of publication,Document type,ProQuest document ID,Document URL,Last updated,first_sentence
1071,ON THIS DATE,None available.,ON THIS DATE Aug. 5 1921: Pittsburgh radio st...,None,Bans; Scandals; Professional baseball,"Chicago Tribune; Chicago, Ill.",2024,"Aug 5, 2024",Sports,"Tribune Publishing Company, LLC","Chicago, Ill.",United States,General Interest Periodicals--United States,10856706,English,News,3087926882,https://www.proquest.com/newspapers/on-this-d...,2024-08-05,ON THIS DATE Aug. 5 1921: Pittsburgh radio st...
1403,Ex-Yank Sheffield behind in the count in Hall...,None available.,With the National Baseball Hall of Fame nearl...,"Phillips, Gary",Drugs & sports; Scandals; Elections; Professi...,"New York Daily News; New York, N.Y.",2024,"Jan 23, 2024",None,"Tribune Publishing Company, LLC","New York, N.Y.",United States,General Interest Periodicals--United States,26921251,English,News,2917461063,https://www.proquest.com/newspapers/ex-yank-s...,2024-01-23,With the National Baseball Hall of Fame nearl...
2034,He's a smooch operator; Boss of the Kiss Cam ...,"On their 10th anniversary, the Los Angeles sp...","While you're cheering for runs and hits, he...","Plaschke, Bill",Professional baseball; Marriage; Weddings,"Los Angeles Times; Los Angeles, Calif.",2016,"Feb 14, 2016",Sports; Part D; Sports Desk,Los Angeles Times Communications LLC,"Los Angeles, Calif.",United States,General Interest Periodicals--United States,04583035,English,News,1764947739,https://www.proquest.com/newspapers/hes-smooc...,2017-11-22,"While you're cheering for runs and hits, he..."
2119,Boston bombing top story,NEW YORK - The Boston Marathon bombing was se...,NEW YORK - The Boston Marathon bombing was ...,"Cohen, Rachel",Marathons; Professional baseball; Athletes; B...,"Telegraph - Herald; Dubuque, Iowa",2013,"Dec 28, 2013",None,Telegraph Herald,"Dubuque, Iowa",United States,General Interest Periodicals--United States,1041293X,English,News,1471067451,https://www.proquest.com/newspapers/boston-bo...,2013-12-29,NEW YORK - The Boston Marathon bombing was ...
2120,Marathon bombing voted top sports story,NEW YORK - The Boston Marathon bombing was se...,NEW YORK - The Boston Marathon bombing was ...,"Cohen, Rachel",Marathons; Athletes; Professional baseball; B...,"Daily Gleaner; Fredericton, N.B.",2013,"Dec 28, 2013",D,Postmedia Network Inc.,"Fredericton, N.B.",Canada,General Interest Periodicals--United States,08216983,English,News,1471010792,https://www.proquest.com/newspapers/marathon-...,2013-12-28,NEW YORK - The Boston Marathon bombing was ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52365,"Author of ""Steroid Nation"" will speak at Brid...","Juiced Home Run Totals, Anti-Aging Miracles, ...","BRIDGEWATER -- Shaun Assael, award-winning ...","of ""Steroid Nation"" will speak at Bridgewater",Professional baseball; Awards & honors,"The News Leader; Staunton, Va.",2010,"Apr 22, 2010",NEWS,Gannett Media Corp,"Staunton, Va.",United States,General Interest Periodicals--United States,None,English,News,440874047,https://www.proquest.com/globalnewsfed/newspa...,2017-11-16,"BRIDGEWATER -- Shaun Assael, award-winning ..."
52498,No urine test for HGH Expert says it would be...,"MLB stopped short of confirming that, saying ...",In a blow to Major League Baseball's and the ...,"Baumbach, Jim; With, Bob",Professional baseball; Urine; Nanoparticles; ...,"Newsday, Combined editions; Long Island, N.Y.",2010,"Feb 25, 2010",SPORTS,Newsday LLC,"Long Island, N.Y.",United States,General Interest Periodicals--United States,None,English,News,283846570,https://www.proquest.com/globalnewsfed/newspa...,2017-11-07,In a blow to Major League Baseball's and the ...
52589,"Who's lying now, Canseco asks: Dares ex-tea...","The informant, who was part of the b

In [13]:
mask = df['Subject'].str.contains('Professional baseball', na=False)
df = df[~mask]

## Topic Cleaning

In [17]:
df['Noise'] = 0
def remove_topic(df:pd.DataFrame, topic_num:int, exceptions:list[int], minor_topic = True):
    if minor_topic:
        mask = (df['topic'] == topic_num) & (~df.index.isin(exceptions))
    else:
        mask = (df['Topic'] == topic_num) & (~df.index.isin(exceptions))
    print(f"The orig length:{len(df)} -> cropped: {len(df[~mask])}")
    return df[~mask]

def mark_noise_topic(df:pd.DataFrame, topic_num:int, exceptions:list[int], minor_topic = True):
    if minor_topic:
        mask = (df['topic'] == topic_num) & (~df.index.isin(exceptions))
    else:
        mask = (df['Topic'] == topic_num) & (~df.index.isin(exceptions))
    df.loc[mask,'Noise'] = 1

# Saving Dataframe

In [53]:
df.to_csv(PROQUEST_PROCESSED_DIR / 'full_df.csv',index=False)